### load packages, data

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(42)

/opt/anaconda3/envs/moe_on_mac/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
# shakespeare dataset

!wget https://raw.githubusercontent.com/AviSoori1x/makeMoE/main/input.txt

--2026-04-01 15:47:20--  https://raw.githubusercontent.com/AviSoori1x/makeMoE/main/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.09s   

2026-04-01 15:47:20 (12.2 MB/s) - ‘input.txt’ saved [1115394/1115394]



### define expert model

In [3]:
class Expert(nn.Module):
    """linear layer followed by relu activation"""

    def __init__(self, n_embd):
        # n_embd denotes embedding dimension
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), # expansion
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), # contraction
            nn.Dropout(dropout=0.1), # 10% dropout, tune later
        )
    
    def forward(self, x):
        return self.net(x)

### implement router

- input matrix: count of tokens x embedding dimension 
- routing matrix: embedding dimension x number of experts
- input matrix X routing matrix = expert selector matrix
- expert selector matrix: tokens x number of experts

- routger determines which expert network receives the output for each token after/from multi-head attention

#### top k-load balancing after

In [ ]:
# gating
num_experts = 3
top_k = 2 # top k experts to select for each token
n_embed = 8

# ex multi-head attention output for example
# consider n_embed=32, context length=4, ...

mh_output = torch.randn(1, 4, n_embed) # batch_size, context_length, embedding_dim

topkgate_linear = nn.Linear(n_embed, num_experts) # router

logits = topkgate_linear(mh_output) # (1, 4, 3)

print(logits) # denotes expert selector

# each row corresponds to one token
# every col corresponds to one expert

tensor([[[ 0.0238, -0.2771, -0.5070],
         [-0.5727, -0.9081,  0.1839],
         [ 0.8137,  0.1781,  1.5661],
         [ 0.6523,  0.4525,  0.0062]]], grad_fn=<ViewBackward0>)


### top_k load balancing

In [ ]:
top_k_logits, top_k_indices = logits.topk(top_k, dim=-1) # (1, 4, 2) obtain top-k experts for each token
top_k_logits, top_k_indices

(tensor([[[ 0.0238, -0.2771],
          [ 0.1839, -0.5727],
          [ 1.5661,  0.8137],
          [ 0.6523,  0.4525]]], grad_fn=<TopkBackward0>),
 tensor([[[0, 1],
          [2, 0],
          [2, 0],
          [0, 1]]]))

### -inf and apply softmax

In [6]:
zeros = torch.full_like(logits, float('-inf')) # (1, 4, 3) create a tensor of -inf values to mask out non-top-k experts
sparse_logits = zeros.scatter(-1, top_k_indices, top_k_logits) # (1, 4, 3) scatter the top-k logits into the zeros tensor
sparse_logits

tensor([[[ 0.0238, -0.2771,    -inf],
         [-0.5727,    -inf,  0.1839],
         [ 0.8137,    -inf,  1.5661],
         [ 0.6523,  0.4525,    -inf]]], grad_fn=<ScatterBackward0>)

In [7]:
gating_output = F.softmax(sparse_logits, dim=-1) # (1, 4, 3) softmax to get gating probabilities
gating_output

tensor([[[0.5747, 0.4253, 0.0000],
         [0.3194, 0.0000, 0.6806],
         [0.3203, 0.0000, 0.6797],
         [0.5498, 0.4502, 0.0000]]], grad_fn=<SoftmaxBackward0>)